In [2]:
import pandas as pd

In [3]:
import numpy as np

In [9]:

df = pd.read_csv('course_lead_scoring_2026.csv')

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [12]:

print(df.shape)
df.info()

(5000, 9)
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [10]:

df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

## Q1. WHAT IS THE MOST FREQUENT OBSERVATION FOR THE COLUMN INDUSTRY

In [ ]:
df['industry'].mode()  # tecnology is the most frequent observation (mode) for the column industry

0    technology
Name: industry, dtype: str

## DATA PREPARATION

In [ ]:
# If there are missing values:
#For categorical features, replace them with 'NA'
#For numerical features, replace with with 0.0

categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Q2.CORRELATION

In [ ]:

numerical_features = df.select_dtypes(include='number')

corr = numerical_features.corr()

corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [16]:

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for col1, col2 in pairs:
    print(col1, 'and', col2, ':', corr.loc[col1, col2]) # *interaction_count and lead_score* are the two features that have the biggest correlation

interaction_count and lead_score : 0.9157457980418697
number_of_courses_viewed and lead_score : 0.7572042392104886
number_of_courses_viewed and interaction_count : 0.7216090038937035
annual_income and interaction_count : 0.12284235135955959


In [17]:

from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=1
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=1
)

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [18]:

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [19]:

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(3000,)
(1000,)
(1000,)


In [20]:

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

In [21]:

from sklearn.feature_extraction import DictVectorizer

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

X_train.shape, X_val.shape

((3000, 28), (1000, 28))

In [22]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, y_pred)
print('AUC:', auc)

AUC: 0.7865162981649108


/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Q3. MUTUAL SCORE

In [ ]:


from sklearn.metrics import mutual_info_score

def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False).round(2)     # lead_source has the biggest mutual information score

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Q4. LOGISTIC REGRESSION AND ACCURACY

In [24]:


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

print("Accuracy:", round(accuracy, 2))  # Accuaracy 0.65

Accuracy: 0.65


## Q5. FEATURE ELIMINATION

In [25]:


from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = categorical + numerical

def train_and_evaluate(features):
    train_dicts = df_train[features].to_dict(orient='records')
    val_dicts = df_val[features].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train_new = dv.fit_transform(train_dicts)
    X_val_new = dv.transform(val_dicts)

    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_new, y_train)
    predictions = model.predict(X_val_new)

    return accuracy_score(y_val, predictions)

original_accuracy = train_and_evaluate(features)

for feature in [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]:
    reduced_features = [f for f in features if f != feature]
    accuracy_without = train_and_evaluate(reduced_features)

    difference = original_accuracy - accuracy_without

    print(
        feature,
        'accuracy without =', accuracy_without,
        'difference =', difference
    )                                             # lead_source has the smallest difference

lead_source accuracy without = 0.648 difference = 0.0030000000000000027
number_of_courses_viewed accuracy without = 0.645 difference = 0.006000000000000005
interaction_count accuracy without = 0.593 difference = 0.05800000000000005


## Q6. REGULAZED LOGISTIC REGRESSION

In [26]:


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    print(f'C = {C}, accuracy = {accuracy:.3f}') # 0.001 has the highest accuracy

C = 1e-06, accuracy = 0.590
C = 1e-05, accuracy = 0.590
C = 0.0001, accuracy = 0.590
C = 0.001, accuracy = 0.648


## ANSWER



| Q | Answer |
|---|---|
| 1 | technology |
| 2 | interaction_count and lead_score |
| 3 | lead_source|
| 4 | 0.65.|
| 5 | lead_source |
| 6 | 0.001 |